<a href="https://colab.research.google.com/github/BenMillerDev/Applied-LLM-Systems/blob/week-1-tokenization/week1_tokenization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 1 (starter): Tokenization Analysis

This is the starter notebook for the Week 1 assignment. It runs as-is on placeholder text so you can see the shape of each step; your job is to replace the placeholders with your own passages and analysis, then commit it to your repository and open a pull request.

Cells marked **TODO (you)** are where you do the work. Everything runs in Jupyter or Google Colab. No GPU, no API key, one dependency: `tiktoken`.

The five parts match the assignment: stand up your repo, run the analysis, evaluate with real figures, find one failure, and submit.

In [1]:
# Setup. In Colab, uncomment the install line on first run.
!pip install tiktoken
import os, pathlib
os.environ['TIKTOKEN_CACHE_DIR'] = str((pathlib.Path('.') / '.tiktoken_cache').resolve())
os.makedirs(os.environ['TIKTOKEN_CACHE_DIR'], exist_ok=True)

import tiktoken
gpt4  = tiktoken.get_encoding('cl100k_base')   # GPT-4 / GPT-3.5
gpt4o = tiktoken.get_encoding('o200k_base')    # GPT-4o
print('tiktoken', tiktoken.__version__, '- encoders ready (cl100k_base, o200k_base)')

tiktoken 0.14.0 - encoders ready (cl100k_base, o200k_base)


## Part 1: Stand up your repository

Do this once, outside the notebook:

1. Create a public repo (suggested name `cosc-650`).
2. Add a `README.md` a stranger could read (what it is, how it is organized, the tools you use).
3. Add an agent context file that your AI tool reads, with project context and conventions. `AGENTS.md` is the cross-tool convention; `CLAUDE.md` and `GEMINI.md` are tool-specific variants. Use whichever your tool reads.
4. Work on a branch and open a pull request into `main`. You will do this every week.

Then commit this notebook into the repo and keep going.

## Helpers (provided)

Two small functions: count tokens for a string, and show the exact sub-token pieces a word breaks into. The demo uses a line you may recognize.

In [2]:
def count_tokens(text, enc):
    return len(enc.encode(text))

def show_split(word, enc=gpt4):
    ids = enc.encode(word)
    pieces = [enc.decode([i]) for i in ids]
    print(f'{word!r:18s} -> {len(ids)} token(s): {pieces}')

# demo: some short strings are a single token; capitalized or rarer words fragment
for w in ['Panic', ' towel', '42', 'antidisestablishmentarianism']:
    show_split(w)

'Panic'            -> 2 token(s): ['P', 'anic']
' towel'           -> 1 token(s): [' towel']
'42'               -> 1 token(s): ['42']
'antidisestablishmentarianism' -> 6 token(s): ['ant', 'idis', 'establish', 'ment', 'arian', 'ism']


## Part 2: Your passages

**TODO (you):** replace the two placeholders with your own text. The non-English passage must be at least 100 words, with a faithful English translation. The placeholders below are short Hitchhiker's Guide lines so the notebook runs; swap in your real passages.

In [3]:
# Opening of a Vietnamese news article marking Ha Long Bay's
# 30th anniversary as a UNESCO World Natural Heritage Site.
# Vietnamese source: https://www.vietnamplus.vn/vinh-ha-long-va-chang-duong-30-nam-duoc-cong-nhan-la-di-san-thien-nhien-the-gioi-post1002064.vnp
# English translated by Claude (Anthropic)
english_text = (
    "December 17, 2024 will be a memorable milestone as Ha Long Bay "
    "completes 30 years since being recognized by UNESCO as a World Natural "
    "Heritage Site. Throughout the past three decades, the magnificent "
    "beauty of Ha Long Bay has been not only the pride of Quang Ninh "
    "province but also of all Vietnam, becoming one of the most attractive "
    "destinations on the planet. With thousands of undulating limestone "
    "islands, mysterious caves, pristine beaches, and a rich ecosystem, Ha "
    "Long Bay deserves to be called a living ink-wash painting, gathering "
    "together the full beauty of nature. This is not only a beautiful "
    "landscape but also a treasure trove of unique geological and "
    "geomorphological value, with a system of caves and limestone grottoes "
    "millions of years old. The scenery at Ha Long Bay changes with each "
    "time of year, bringing visitors an experience that is always fresh "
    "and fascinating."
)

foreign_text = (
    "Ngày 17/12/2024 sẽ là cột mốc đáng nhớ khi Vịnh Hạ Long tròn 30 năm "
    "được UNESCO công nhận là Di sản thiên nhiên thế giới. Trong suốt ba "
    "thập kỷ qua, vẻ đẹp kỳ vĩ của Vịnh Hạ Long không chỉ là niềm tự hào "
    "của tỉnh Quảng Ninh mà còn của cả Việt Nam, trở thành một trong những "
    "điểm đến hấp dẫn nhất hành tinh. Với hàng nghìn đảo đá vôi nhấp nhô, "
    "những hang động kỳ bí, bãi biển hoang sơ và hệ sinh thái phong phú, "
    "Vịnh Hạ Long xứng đáng là một bức tranh thủy mặc sống động, quy tụ "
    "đầy đủ vẻ đẹp của thiên nhiên. Đây không chỉ là một cảnh quan tuyệt "
    "đẹp mà còn là một kho tàng giá trị địa chất, địa mạo độc đáo với hệ "
    "thống hang động, động đá vôi hàng triệu năm tuổi. Cảnh sắc tại Vịnh "
    "Hạ Long thay đổi theo từng thời điểm trong năm, mang đến cho du khách "
    "một trải nghiệm luôn mới mẻ và kỳ thú."
)

print('English words:', len(english_text.split()))
print('Foreign words:', len(foreign_text.split()))

def report(label, text):
    print(f'{label:9s} | chars {len(text):4d} | GPT-4 {count_tokens(text, gpt4):4d} | GPT-4o {count_tokens(text, gpt4o):4d}')

report('English', english_text)
report('Foreign', foreign_text)

tax_gpt4  = count_tokens(foreign_text, gpt4)  / count_tokens(english_text, gpt4)
tax_gpt4o = count_tokens(foreign_text, gpt4o) / count_tokens(english_text, gpt4o)
print(f'\nMultilingual tax  GPT-4: {tax_gpt4:.2f}x   GPT-4o: {tax_gpt4o:.2f}x')

English words: 147
Foreign words: 176
English   | chars  887 | GPT-4  177 | GPT-4o  176
Foreign   | chars  790 | GPT-4  395 | GPT-4o  232

Multilingual tax  GPT-4: 2.23x   GPT-4o: 1.32x


### TODO: One or two sentences interpreting these numbers for YOUR language pair
The Vietnamese text has more words but fewer characters since Vietnamese is a mono-syllabic language.  Most of the words are short and there are spaces between each word.
GPT-4o tokenizes Vietnamese much more effectively, which suggests it recognizes more Vietnamese syllables as single units.

## Part 3: Evaluate with real figures

Turn the counts into engineering consequences. The skeleton below computes both; keep it pointed at your real passages.

In [4]:
CTX = 128_000
en = count_tokens(english_text, gpt4)
fo = count_tokens(foreign_text, gpt4)
print(f'A {CTX:,}-token window holds about {CTX//en:,} English copies and {CTX//fo:,} foreign copies of your passage.')
print(f'Per-request cost multiplier for the foreign language: {fo/en:.2f}x (billing is per token).')
# TODO (you): state what this means for a product serving users in your chosen language.
# For Vietnamese users, the token window has less than half the capacity compared to English.
# This means there would be less than half the space for conversation history and prompts would hit their size limit over twice as fast.
# Under per-token billing, Vietnamese users would pay 2.23x more for the same tasks.
# GPT-4o would be a much more cost-effective choice for Vietnamese users since its cost multiplier was only 1.32x.

A 128,000-token window holds about 723 English copies and 324 foreign copies of your passage.
Per-request cost multiplier for the foreign language: 2.23x (billing is per token).


### TODO: State what this means for a product serving users in your chosen language.
For Vietnamese users, the token window has less than half the capacity compared to English. This means there would be less than half the space for conversation history and prompts would hit their size limit over twice as fast.
Under per-token billing, Vietnamese users would pay 2.23x more for the same tasks. GPT-4o would be a much more cost-effective choice for Vietnamese users since its cost multiplier was only 1.32x.

## Part 4: Bias splits and one failure

**TODO (you):** (a) pick three words where your non-English form fragments far worse than the English equivalent, and show both with `show_split`; (b) find ONE input whose token count defies intuition and explain it. A few failure candidates are demonstrated below to get you started; replace them with your own find and write the explanation plus a mitigation.

In [6]:
# Batch-test bias-split candidates: use pairs from Part 2 text, then sort by the size of the gap to find the strongest three.
candidates = [
    ("nghìn", "thousand"),
    ("tuyệt đẹp", "gorgeous"),
    ("kỳ vĩ", "magnificent"),
    ("hệ sinh thái", "ecosystem"),
    ("địa chất", "geological"),
    ("hoang sơ", "pristine"),
    ("công nhận", "recognized"),
    ("thiên nhiên", "nature"),
    ("di sản", "heritage"),
]

results = []
for vi, en in candidates:
    vi_tokens = count_tokens(vi, gpt4)
    en_tokens = count_tokens(en, gpt4)
    results.append((vi, en, vi_tokens, en_tokens, vi_tokens - en_tokens))

results.sort(key=lambda r: r[4], reverse=True)

print(f'{"Vietnamese":16s} {"English":12s} {"VI tok":6s} {"EN tok":6s} {"gap":4s}')
for vi, en, vt, et, gap in results:
    print(f'{vi:16s} {en:12s} {vt:6d} {et:6d} {gap:4d}')

Vietnamese       English      VI tok EN tok gap 
tuyệt đẹp        gorgeous          8      3    5
thiên nhiên      nature            6      1    5
kỳ vĩ            magnificent       6      3    3
hệ sinh thái     ecosystem         6      3    3
địa chất         geological        5      2    3
công nhận        recognized        4      1    3
nghìn            thousand          4      2    2
hoang sơ         pristine          4      2    2
di sản           heritage          2      2    0


In [15]:
# (a) TODO: three real bias pairs from your languages.
print ('(a) three bias pairs comparing Vietnamese vs English')
print('Bias pair 1: thiên nhiên vs nature')
show_split('thiên nhiên')
show_split('nature')

print('\nBias pair 2: công nhận vs recognized')
show_split('công nhận')
show_split('recognized')

print('\nBias pair 3: tuyệt đẹp vs gorgeous')
show_split('tuyệt đẹp')
show_split('gorgeous')

print('\n(b) failure candidates to explore (replace with your own find):')
show_split('đẹp')                 # translates to "pretty"
show_split('tuyệt đẹp')           # translates to "gorgeus"

import unicodedata
print('\nWord comparison for "tuyệt đẹp"')
word = 'tuyệt đẹp'
# NFC (precomposed) vs NFD (decomposed)
# same visible text, different underlying bytes.
nfc = unicodedata.normalize('NFC', word)
nfd = unicodedata.normalize('NFD', word)
print('NFC form:')
show_split(nfc)
print('NFD form:')
show_split(nfd)

print('\nLetter comparison')
chars = ['ệ', 'ẹ']

for ch in chars:
    print(f'--- {ch!r} ---')
    nfc = unicodedata.normalize('NFC', ch)
    nfd = unicodedata.normalize('NFD', ch)
    print('NFC:')
    show_split(nfc)
    print('NFD:')
    show_split(nfd)
    print()

(a) three bias pairs comparing Vietnamese vs English
Bias pair 1: thiên nhiên vs nature
'thiên nhiên'      -> 6 token(s): ['th', 'i', 'ên', ' n', 'hi', 'ên']
'nature'           -> 1 token(s): ['nature']

Bias pair 2: công nhận vs recognized
'công nhận'        -> 4 token(s): ['c', 'ông', ' nh', 'ận']
'recognized'       -> 1 token(s): ['recognized']

Bias pair 3: tuyệt đẹp vs gorgeous
'tuyệt đẹp'        -> 8 token(s): ['t', 'uy', 'ệ', 't', ' đ', '�', '�', 'p']
'gorgeous'         -> 3 token(s): ['gor', 'ge', 'ous']

(b) failure candidates to explore (replace with your own find):
'đẹp'              -> 4 token(s): ['đ', '�', '�', 'p']
'tuyệt đẹp'        -> 8 token(s): ['t', 'uy', 'ệ', 't', ' đ', '�', '�', 'p']

Word comparison for "tuyệt đẹp"
NFC form:
'tuyệt đẹp'        -> 8 token(s): ['t', 'uy', 'ệ', 't', ' đ', '�', '�', 'p']
NFD form:
'tuyệt đẹp'     -> 12 token(s): ['t', 'uye', '�', '�', '�', '�', 't', ' đ', 'e', '�', '�', 'p']

Letter comparison
--- 'ệ' ---
NFC:
'ệ'                -

## TODO: Explain WHY your chosen case behaves this way, and how you would budget or normalize around it.

I found failure cases with accented letters in Vietnamese.  The two examples I used are "ẹ" and "ệ", and they are each handled differently in NFC vs NFD form.

- "ẹ" produces two tokens in NFC and three tokens in NFD.
- "ệ" produces one token in NFC and five tokens in NFD.

This happens because GPT-4's tokenizer works at the byte level, not the character level. In NFC form, "ệ" is stored as a single Unicode codepoint, and that codepoint's bytes happen to match a token the tokenizer already knows, so it stays as one token. "ẹ" is also a single codepoint in NFC, but its bytes were never learned as a single token, so two of its bytes come out as invalid UTF-8 fragments (shown as "�").  In NFD form, both letters are broken into a base letter ("e") plus one or two separate combining marks. The tokenizer has no learned token for these combining marks either, so their bytes fragment into "�" pieces too. This is why NFD costs more tokens than NFC for both letters.

To budget or normalize around this in production, all incoming text should be converted to NFC before it is tokenized.  This would keep token counts consistent since the same visible word would be a larger number of tokens in NFD form.


## Part 5: Submit

Before you open the pull request, check:

- The notebook runs top to bottom on **your** passages, not the placeholders.
- Your three bias splits are shown and explained.
- The failure case has a cause and a mitigation.
- The PR description has a one-paragraph result summary with your headline numbers.
- You linked one issue in your repo logging this as a research note (title, inputs, what you found).

Rubric: repo quality (15), counts from both tokenizers (20), tax computed (15), three bias splits (20), cost and context figures (15), the failure case (10), PR hygiene (5).